# Selection of Patient Sequences xAI berner 2025

- this file provides an overview of the datastructure of the Data Recieved from BERN
- Manual Preselection for Sequences is provided by Hänsel Martin, Julian Deseö, Hakim Bazoui
- output of this Selection is a csv file with selected DWI sequeces as well as anatomic t1 sewquences per patient for further processing

In [1]:
import os
import pandas as pd
from datetime import datetime
import pydicom
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import csv
import re
from venn import venn


from utils.selection import *


# Source of RAW DICOMs
BERN_PATH='/nasdata/Berner_Daten/Other_Side_Bilddaten' 
# Project data path
ANALYSIS_PATH='/home/bule/berner_daten_analyse/data'

- search all files, filter those with no useful content

In [ ]:
print( f"distintict patient #:   {len(set(os.listdir(BERN_PATH)))}")
if not os.path.exists(os.path.join(ANALYSIS_PATH,"all_dicoms.csv")) or not os.path.exists(os.path.join(ANALYSIS_PATH,"other_files.csv")):
    
    dcm_files, other_files = collect_all_file_paths(BERN_PATH) # takes some time
    # already wrote these.. just loading
    write_list_to_csv(all_dcm_files, os.path.join(ANALYSIS_PATH,"all_dicoms.csv"))
    write_list_to_csv(other_files, os.path.join(ANALYSIS_PATH,"other_files.csv"))
else:
    # onyl dicoms
    dicom_files = read_file_paths_from_csv(os.path.join(ANALYSIS_PATH,"all_dicoms.csv"))
    # only other files
    other_files = read_file_paths_from_csv(os.path.join(ANALYSIS_PATH,"other_files.csv"))

print(f"Total  .dcm files: {len(dicom_files)}")
print(f"Tota l other files: {len(other_files)}")

# ALL DICOM FILES

create dataframe with entries for each path

In [ ]:
if not os.path.exists(os.path.join(ANALYSIS_PATH,"dicom_files_df.csv")):
    data=[path.split('/')[4:] for path in dicom_files]
    for sublist, path in zip(data, dicom_files):
        sublist.append(path)
    columns = ['Patient_ID', 'Study_ID', 'Sequence_ID', 'File_Name', 'Full_Path']
    df = pd.DataFrame(data, columns=columns)
    df['Date'] = df['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))
    df['Sequence_ID_split'] = df['Sequence_ID'].apply(lambda x: split_at_anynumber_seq(x))
    df.to_csv(os.path.join(ANALYSIS_PATH,"dicom_files_df.csv"), index=False)
    
else:
    print("dicom_files_df.csv already exists")
    df = pd.read_csv(os.path.join(ANALYSIS_PATH,"dicom_files_df.csv"),sep=',')
    print()
    
# SAVE ONLY PATHS  for later use
df_only_paths = df['Full_Path']

if not os.path.exists(os.path.join(ANALYSIS_PATH,"only_dcm_paths.csv")):
    df_only_paths.to_csv(os.path.join(ANALYSIS_PATH,"only_dcm_paths.csv"), index=False)
else:
    print("only_dcm_paths.csv already exists")
    
print(df.info())
df.head()

### Stats


- alle __ fangen mit 1.2.* an
- die mit eintrag benannten sequenzen mit ** .1.3.  * 

#### Sequence ID

- 100_146 differnet sequence names

In [ ]:
list(set(df['Sequence_ID']))

- split files at __1.3 XY and only keep the first part

In [ ]:
# def split_at_anynumber(sequences):
#     return [re.split(r'__\d+\.', seq, maxsplit=1)[0] for seq in sequences]

# cleaned_sequences = split_at_anynumber(list(set(df['Sequence ID'])))
# cleaned_set=set(cleaned_names)
# cleaned_set

df = pd.read_csv(os.path.join(ANALYSIS_PATH,"dicom_files_df.csv"),sep=',')
df.head()

- sequence_ID_split is the Sequence_ID but with remove after __*

## First filtering 

- remove all CT scans
- remove all sagital and coronal scan

In [ ]:
unique_sequences=list(set(df['Sequence_ID_split']))
print(len(unique_sequences))

In [ ]:
# remove CT in the NAME not by inspecting dicom - > inspecting takes a long time for many files
unique_sequences=sorted([str(x) for x in unique_sequences if 'CT' not in str(x)])
print(len(unique_sequences))

In [ ]:
# COR
unique_sequences=sorted([str(x) for x in unique_sequences if 'coronal' not in str(x).lower()])
print(len(unique_sequences))

unique_sequences=sorted([str(x) for x in unique_sequences if 'coronar' not in str(x).lower()])
print(len(unique_sequences))

pattern = r"[^a-zA-Z0-9]cor[^a-zA-Z0-9]"
unique_sequences = [s for s in unique_sequences if not  re.search(pattern, s.lower())]
print(len(unique_sequences))


unique_sequences = [s for s in unique_sequences if not  s.lower().endswith('cor')]
print(len(unique_sequences))

unique_sequences = [s for s in unique_sequences if not  s.lower().startswith('cor')]
print(len(unique_sequences))

unique_sequences=sorted([str(x) for x in unique_sequences if 'coro' not in str(x).lower()])
print(len(unique_sequences))

In [ ]:
# SAG
pattern = r"[^a-zA-Z0-9]sag[^a-zA-Z0-9]"
unique_sequences = [s for s in unique_sequences if not  re.search(pattern, s.lower())]
print(len(unique_sequences))

unique_sequences=sorted([str(x) for x in unique_sequences if 'sagittal' not in str(x).lower()])
print(len(unique_sequences))

unique_sequences=sorted([str(x) for x in unique_sequences if 'sagital' not in str(x).lower()])
print(len(unique_sequences))

unique_sequences = [s for s in unique_sequences if not  s.lower().endswith('sag')] 
print(len(unique_sequences))

unique_sequences = [s for s in unique_sequences if not  s.lower().startswith('sag')] 
print(len(unique_sequences))

#### mit Hakim,Julian und Martin durchgeschaute begriffe für Auschluss Online Meeting vom  3.1.25  

In [ ]:
# selections for the full sequence
lower=['angio','carotis','carotid','abdomen','becken','aorta','felsenbein','knochen','thorax','lunge','perfusion','arteriel','venoes','mip','mtt','tmax','ttp','tissue','time to peak','bold'
       'weichteile','wilisi','willis','vessel']

unique_sequences = [
    seq for seq in unique_sequences 
    if not any(keyword.lower() in seq.lower() for keyword in lower)
]
len(unique_sequences)

In [ ]:
# distinct sequences
not_Case=['CBF','CBV','DSA','Th','HWS','Herz','GADO','KM','Abd','Blood','COW','Bone','Gesicht','ICA','Kopf_Hals','Infiltr.','TOF','MTT','Transit','transit','MIP','Mediastinum','NOVA','RMCA','LMCA','MPR','hws','karotis','tof','postinterventionelle Kontrolle','ToF','Tof','OLEA_ANALYSIS']
unique_sequences = [
    seq for seq in unique_sequences 
    if not any(keyword in seq for keyword in not_Case)
]
len(unique_sequences)

- add a bool for if sewuence is in selction

In [12]:
df['Sequence_ID_split_selection'] = df['Sequence_ID_split'].apply(lambda x: x in unique_sequences) #~ 50 s
df_sorted = df.sort_values(by=["Sequence_ID", "File_Name"])

- read dicom files and extract metadata
- remove files which have CT

In [ ]:

if not os.path.exists(os.path.join(ANALYSIS_PATH,"selected_sequences_names_all1.csv")):
    
    splifirst_row_per_groupt_selection=df_sorted[df_sorted['Sequence_ID_split_selection']==True].groupby("Sequence_ID").first().reset_index()
    splifirst_row_per_groupt_selection
    
    # reorder columns
    column_order = ["Patient_ID", "Study_ID", "File_Name", "Full_Path",  "Date","Sequence_ID", "Sequence_ID_split", "Sequence_ID_split_selection"]
    splifirst_row_per_groupt_selection= splifirst_row_per_groupt_selection[column_order]
    splifirst_row_per_groupt_selection
    # takes some time to read all teh 29855 dicom files ~ 5 min # would be around 35 days to calculate in this way for all 5M dicom files
    splifirst_row_per_groupt_selection['Modality'] = splifirst_row_per_groupt_selection.apply(
    lambda x: get_dicom_value(x['Full_Path']), axis=1)
    splifirst_row_per_groupt_selection.to_csv(os.path.join(ANALYSIS_PATH,"selected_sequences_names_all1.csv"), index=False)
else:
    splifirst_row_per_groupt_selection = pd.read_csv(os.path.join(ANALYSIS_PATH,"selected_sequences_names_all1.csv"),sep=',')
    print("selected_sequences_names_all1.csv already exists")
    
    
# remnove CT
removed_ct=splifirst_row_per_groupt_selection[splifirst_row_per_groupt_selection['Modality']!='CT']
removed_ct.head()

#### this file was sent to Martin Hänsler HAkim Bazoui and Julian Desöe for a positiv selection

In [ ]:
# rem
removed_first=removed_ct.groupby("Sequence_ID_split").first().reset_index()
removed_first_selection=removed_first[['Full_Path', 'Sequence_ID_split']]
removed_first_selection['Updated_Path'] = removed_first_selection['Full_Path'].apply(
    lambda x: os.path.join(*x.split('/')[3:])
)
removed_first_selection

### read in selections from hakim, julian and martin - Date 7.9.25

Selection asccordind to the numbers
-  1  T1
-  2  T2
-  3  FLAIR
-  4  DWI     #Comemnt: JD also marked ADC as nr 4 
-  5  Unklar («spc», heisst das SPACE und wäre analog der FLAIR?)

In [ ]:
removed_first=removed_ct.groupby("Sequence_ID_split").first().reset_index()
selection_df = removed_first.loc[:, ~removed_first.columns.isin(['Sequence_ID_split_selection'])]
selection_df

In [ ]:
# read the 3 seletkoin files
df_hakim = pd.read_csv(os.path.join(ANALYSIS_PATH,"selected_sequences_names_distinct_HB.csv"),sep=',')
df_martin = pd.read_csv(os.path.join(ANALYSIS_PATH,"selected_sequences_names_distinct_MH.csv"),sep=';')
df_julian = pd.read_csv(os.path.join(ANALYSIS_PATH,"selected_sequences_names_distinct_JD.csv"),sep=',')

# merge the 3 files / fill na and rename columns
merge1=selection_df.merge(df_hakim[['Sequence_ID','HB_selektion']], on='Sequence_ID', how='left')
merge2=merge1.merge(df_martin[['Sequence_ID_split','Code','most likely']], on='Sequence_ID_split', how='left')
merge2=merge2.rename(columns={"Code": "MH_selektion", "most likely": "MH_most_likely"})
merge2['HB_selektion'] = merge2['HB_selektion'].fillna(0.0)
merge3=merge2.merge(df_julian[['Sequence_ID_split','Sequenz']].fillna(0.0),on='Sequence_ID_split')
merge3.rename(columns={"Sequenz": "JD_selektion"}, inplace=True)
merge3.head()

- agreements and disagreemnet

In [17]:
discrepancy_df = merge3[merge3[['HB_selektion', 'MH_selektion', 'JD_selektion']].nunique(axis=1) > 1]
intersection_df = merge3[merge3[['HB_selektion', 'MH_selektion', 'JD_selektion']].nunique(axis=1) == 1]

In [18]:
discrepancy_df.to_csv(os.path.join(ANALYSIS_PATH,"discrepancy_df_9_01_25.csv"), index=False)

leftovoder postive selected:

In [ ]:
intersection_df_nonzero=intersection_df[intersection_df['HB_selektion']!=0][['Sequence_ID_split','HB_selektion']] # is set
print(f"intersection_df_nonzero: {len(intersection_df_nonzero)}")
intersection_df_nonzero.head()

- add agreemnet numbers to the big dataframe

In [ ]:
mapping = dict(zip(intersection_df_nonzero['Sequence_ID_split'], intersection_df_nonzero['HB_selektion']))
df['HB_JD_HB_selection'] = df['Sequence_ID_split'].map(mapping).fillna(0.0)
df.head()

### See intersection of sequences

- dwi only earliest dates

In [ ]:
#
dwi_df= df[df['HB_JD_HB_selection']==4]
# dwi_df = dwi_df.sort_values(by=['Patient_ID', 'Date']).reset_index(drop=True)
# dwi_unique_df = dwi_df.groupby('Patient_ID').first().reset_index()
# len(dwi_unique_df)

earliest_dates = dwi_df.groupby('Patient_ID')['Date'].min()
dwi_df = dwi_df.merge(earliest_dates, on=['Patient_ID', 'Date'])
dwi_unique_df = dwi_df.groupby('Patient_ID').first().reset_index()
len(dwi_unique_df)

In [ ]:
t1_df= df[df['HB_JD_HB_selection']==1]
t1_df = t1_df.sort_values(by=['Patient_ID', 'Date']).reset_index(drop=True)
t1_unique_df =t1_df.groupby('Patient_ID').first().reset_index()
len(t1_unique_df)

#t1_df.to_csv(os.path.join(ANALYSIS_PATH,"t1_df.csv"), index=False)

In [ ]:
t2_df= df[df['HB_JD_HB_selection']==2]
t2_df = t2_df.sort_values(by=['Patient_ID', 'Date']).reset_index(drop=True)
t2_unique_df =t2_df.groupby('Patient_ID').first().reset_index()
len(t2_unique_df)

In [ ]:
flair_df= df[df['HB_JD_HB_selection']==3]
flair_df = flair_df.sort_values(by=['Patient_ID', 'Date']).reset_index(drop=True)
flair_unique_df =flair_df.groupby('Patient_ID').first().reset_index()
len(flair_unique_df)

In [ ]:
t1_t2_flair_df = df[(df['HB_JD_HB_selection'] == 1) | 
                    (df['HB_JD_HB_selection'] == 2) | 
                    (df['HB_JD_HB_selection'] == 3)]
t1_t2_flair_df = t1_t2_flair_df.sort_values(by=['Patient_ID', 'Date']).reset_index(drop=True)
t1_t2_flair_unique_df = t1_t2_flair_df.groupby('Patient_ID').first().reset_index()
len(t1_t2_flair_unique_df)

- overlaps

In [ ]:
data = {
    'T1': set(t1_unique_df['Patient_ID']),
    'T2': set(t2_unique_df['Patient_ID']),
    'FLAIR': set(flair_unique_df['Patient_ID']),
    'DWI': set(dwi_unique_df['Patient_ID'])
}
set_labels = {
    'T1': data['T1'],
    'T2': data['T2'],
    'FLAIR': data['FLAIR'],
    'DWI': data['DWI']
}
venn(set_labels)
plt.title("Venn Diagram for T1, T2, FLAIR, and DWI")
plt.show()

- dwi has most overlap with T1 : 228 , addition al sequences only add 10

- create Dictionary per patient with all sequences dwi and T1
- count how many exist per dwi and t1

In [ ]:
intersection_pat_ids_dwi_t1=set(t1_unique_df['Patient_ID']).intersection(set(dwi_unique_df['Patient_ID']))

dict_t1_dwi={key:{} for key in intersection_pat_ids_dwi_t1}

for key in intersection_pat_ids_dwi_t1:
    dict_t1_dwi[key]['T1']=list(t1_df[t1_df['Patient_ID']==key]['Full_Path'])
    dict_t1_dwi[key]['DWI']=list(dwi_df[dwi_df['Patient_ID']==key]['Full_Path'])

data = []
for pat_id, dictionaries in dict_t1_dwi.items():
    n_t1=len(dictionaries['T1'])
    n_dwi=len(dictionaries['DWI']) 
    data.append({"Patient ID": pat_id, "T1 Paths Count": n_t1, "DWI Paths Count": n_dwi})
count_dwi_t1_first = pd.DataFrame(data)
count_dwi_t1_first

- stats min max median

In [ ]:
count_dwi_t1_first[['T1 Paths Count','DWI Paths Count']].describe()

- save as df

In [29]:
dwi_t1_all= []
for pat_id, dictionaries in dict_t1_dwi.items():
    
    for t1_path in dictionaries['T1']:
            dwi_t1_all.append({ "Full_Path": t1_path, "Modality": 't1'})
    for dwi_path in dictionaries['DWI']:
            dwi_t1_all.append({ "Full_Path": dwi_path, "Modality": 'dwi'})  
dwi_t1_all_df = pd.DataFrame(dwi_t1_all)

dwi_t1_all_df_merge=df.merge(dwi_t1_all_df, on='Full_Path', how='right')




dwi_t1_all_df_merge.to_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_all.csv"), index=False)   

- this file is exported 

# ----- INSPECT THE other FILES ---------------------------------------------------



- one patient has a lot of additoin al DICOMS PAT 0010822127

## IMAGES

- 201  file with: 


has some special naming schema, Also DICOMS
['0010822127', '20120121_2840853', '0', 'IMAGES', 'SRS0', 'IMG0', '/nasdata/Berner_Daten/Other_Side_Bilddaten/0010822127/20120121_2840853/0/IMAGES/SRS0/IMG0']

In [ ]:
file_path = '/nasdata/Berner_Daten/Other_Side_Bilddaten/0010822127/20120121_2840853/0/IMAGES/SRS0/IMG0'
if os.path.isfile(file_path):
    file_type = magic.from_file(file_path)
    print(f"File: {file_path}")
    print(f"File Type: {file_type}")
else:
    print(f"The file does not exist: {file_path}")
    
dicom_data = pydicom.dcmread(file_path, stop_before_pixels=True)  # Load without pixel data to check header only
dicom_data

In [ ]:
other_files = read_file_paths_from_csv("/home/bule/berner_daten_analyse/other_files.csv")
columns = ['Patient ID', 'Study ID', 'Subfolder','IMAGES_DIR','SRS', 'FILENAME', 'Full Path']


other_data=[path.split('/')[4:] for path in other_files if 'IMAGES' in path]
print(len(other_data))
for sublist in other_data:
    sublist.append(os.path.join(BERN_PATH, *sublist))

print((other_data[0]))

df_others_images = pd.DataFrame(other_data, columns=columns)
df_others_images['Date'] = df_others_images['Study ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))
print(    f" unique ids {df_others_images['Patient ID'].nunique()} , ids {df_others_images['Patient ID'].unique()}")
df_others_images.head()


In [ ]:
df_others_images['Patient ID'].nunique()

## PNG

- 201  files with .pngs

In [ ]:
img = mpimg.imread('/nasdata/Berner_Daten/Other_Side_Bilddaten/0010822127/20120121_2840853/PNG/ADC/ADC_00.png')
plt.imshow(img)
plt.show()

In [ ]:
other_files = read_file_paths_from_csv("/home/bule/berner_daten_analyse/other_files.csv")
columns = ['Patient ID', 'Study ID', 'Subfolder','MOD', 'FILENAME', 'Full Path']

other_data=[path.split('/')[4:] for path in other_files if 'PNG' in path]
print(len(other_data))
for sublist in other_data:
    sublist.append(os.path.join(BERN_PATH, *sublist))
print((other_data[0]))

df_others_PNG = pd.DataFrame(other_data, columns=columns)
df_others_PNG['Date'] = df_others_PNG['Study ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))
print(    f" unique ids {df_others_PNG['Patient ID'].nunique()} , ids {df_others_PNG['Patient ID'].unique()}")
df_others_PNG.head()


## DICOMDIR
only one file  for one pat. : 0010822127

In [ ]:
other_files = read_file_paths_from_csv("/home/bule/berner_daten_analyse/other_files.csv")
columns = ['Patient ID', 'Study ID', 'Subfolder', 'FILENAME', 'Full Path']

other_data=[path.split('/')[4:] for path in other_files if 'DICOMDIR' in path]
print(len(other_data))
for sublist in other_data:
    sublist.append(os.path.join(BERN_PATH, *sublist))
print((other_data[0]))

df_others_DICOMDIR = pd.DataFrame(other_data, columns=columns)
df_others_DICOMDIR['Date'] = df_others_DICOMDIR['Study ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))
df_others_DICOMDIR.head()

In [ ]:
file_path = '/nasdata/Berner_Daten/Other_Side_Bilddaten/0010822127/20120121_2840853/0/DICOMDIR'
if os.path.isfile(file_path):
    file_type = magic.from_file(file_path)
    print(f"File: {file_path}")
    print(f"File Type: {file_type}")
else:
    print(f"The file does not exist: {file_path}")
    
dicom_data = pydicom.dcmread(file_path, stop_before_pixels=True)  
dicom_data